# Z-Image on Kaggle TPU v5e-8 — Production Showcase

## Before you run / Trước khi chạy

**English**

This is the public research showcase for the full, undistilled `Tongyi-MAI/Z-Image` model on a Kaggle TPU v5e-8. Use **Add Input → Models** to attach [Tongyi-MAI/Z-Image](https://www.kaggle.com/models/dangkhoa2016/tongyi-mai-z-image) (PyTorch/default/version 1) and verify its mount; enable Internet for the source refresh, select TPU v5e-8, then use **Run All**. The default path produces 14 real images (two independent 512×512 samples, four 1024×1024 technical images, and eight 1024×1024 creative images). High-resolution inference is optional and adds one fifteenth image.

**Tiếng Việt**

Đây là showcase nghiên cứu v1.0.0 công khai của model `Tongyi-MAI/Z-Image` full, undistilled trên Kaggle TPU v5e-8. Dùng **Add Input → Models** để attach [Tongyi-MAI/Z-Image](https://www.kaggle.com/models/dangkhoa2016/tongyi-mai-z-image) (PyTorch/default/version 1), kiểm tra model mount, bật Internet để refresh source, chọn TPU v5e-8 rồi dùng **Run All**. Mặc định notebook tạo 14 ảnh thật (hai ảnh 512×512 độc lập, bốn ảnh kỹ thuật 1024×1024 và tám ảnh creative 1024×1024). Inference độ phân giải cao là tùy chọn và thêm ảnh thứ 15.


### Attach the model / Gắn model vào Notebook

**English — Kaggle setup**

1. Open [Z-Image on Kaggle Models](https://www.kaggle.com/models/dangkhoa2016/tongyi-mai-z-image).
2. In the Kaggle Notebook editor, open **Add Input** (right sidebar), choose **Models**, and find **dangkhoa2016/tongyi-mai-z-image**.
3. Attach variation **PyTorch → default → version 1**. Wait for the model to appear under Notebook Inputs.
4. The read-only model directory should be `/kaggle/input/models/dangkhoa2016/tongyi-mai-z-image/pytorch/default/1`. Check that `model_index.json`, `transformer/`, `vae/`, and `text_encoder/` are present.
5. In Notebook settings, select **TPU v5e-8** and enable **Internet**. In **Cell 7 — Runtime configuration**, change `ZIMAGE_RUN_2000PX = False` to `True` if you want the optional 2048×1904 image. Then click **Run All**.

**Tiếng Việt — Thiết lập Kaggle**

1. Mở [Z-Image trên Kaggle Models](https://www.kaggle.com/models/dangkhoa2016/tongyi-mai-z-image).
2. Trong Kaggle Notebook, mở **Add Input** (thanh bên phải), chọn **Models**, tìm **dangkhoa2016/tongyi-mai-z-image**.
3. Attach variation **PyTorch → default → version 1** và đợi model xuất hiện trong Notebook Inputs.
4. Thư mục model chỉ đọc dự kiến: `/kaggle/input/models/dangkhoa2016/tongyi-mai-z-image/pytorch/default/1`. Kiểm tra `model_index.json`, `transformer/`, `vae/`, `text_encoder/`.
5. Trong Notebook settings, chọn **TPU v5e-8**, bật **Internet**. Tại **cell cấu hình runtime (cell mã thứ ba)**, đổi `ZIMAGE_RUN_2000PX = False` thành `True` nếu muốn tạo thêm ảnh 2048×1904. Sau đó chọn **Run All**.

**English:** Notebook image previews have a maximum edge of 768 px. Full resolution PNGs are available in the live Cloudflared gallery.

**Tiếng Việt:** Ảnh xem trước có cạnh tối đa 768 px để giảm dung lượng notebook và giảm tải trình duyệt. PNG nguyên bản được phục vụ tại gallery Cloudflared.

In [ ]:
import time as _run_clock, os as _run_os
RUN_WALL_START = _run_clock.time()
PERFORMANCE_TARGET_SECONDS = float(_run_os.environ.get('ZIMAGE_PERFORMANCE_TARGET_SECONDS', '1200'))

from pathlib import Path
import os, subprocess
REPO_URL = "https://github.com/dangkhoa2016/Z-Image-On-Kaggle-TPU-v5e8.git"
BASE = Path(os.environ.get("ZIMAGE_REPO_DIR", "/kaggle/working/zimage-tpu")).expanduser().resolve()
SOURCE_REF = os.environ.get("ZIMAGE_SOURCE_REF", "main")
if (BASE / ".git").exists():
    local_ref = subprocess.run(["git", "rev-parse", "--verify", SOURCE_REF + "^{commit}"], cwd=BASE,
                               text=True, capture_output=True)
    if SOURCE_REF == "main" or local_ref.returncode != 0:
        subprocess.run(["git", "fetch", "origin", "--tags", "--force"], cwd=BASE, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(BASE)], check=True)
subprocess.run(["git", "checkout", "--detach", "origin/main" if SOURCE_REF == "main" else SOURCE_REF], cwd=BASE, check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=BASE, text=True).strip()
status = subprocess.check_output(["git", "status", "--porcelain"], cwd=BASE, text=True)
assert not status.strip(), "Refusing to run from a dirty public-source checkout"
print("SOURCE_HEAD", head)
print("SOURCE_STATUS clean")


## 1. Dependency bootstrap / Cài dependency

**English**

Install the repository's pinned user-space dependencies from `requirements.txt` while preserving Kaggle's preinstalled PyTorch/XLA stack. This makes a fresh imported notebook self-contained instead of relying on packages installed in an earlier session.

**Tiếng Việt**

Cài các user-space dependency đã pin từ `requirements.txt` nhưng giữ nguyên PyTorch/XLA có sẵn của Kaggle. Nhờ vậy notebook import mới có thể tự chạy, không phụ thuộc package đã cài từ session trước.


In [ ]:
import sys
requirements = BASE / "requirements.txt"
assert requirements.exists(), requirements
dependency_install = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements), "--root-user-action=ignore"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
LOG_BOOTSTRAP = BASE / "evidence" / "runtime" / "notebook-logs"
LOG_BOOTSTRAP.mkdir(parents=True, exist_ok=True)
(LOG_BOOTSTRAP/"dependency-install.log").write_text(dependency_install.stdout or "", encoding="utf-8")
if dependency_install.returncode:
    print(dependency_install.stdout[-2500:])
    raise subprocess.CalledProcessError(dependency_install.returncode, "pip install requirements")
print("DEPENDENCIES_READY", requirements, "raw_log", LOG_BOOTSTRAP/"dependency-install.log")


# Each subprocess keeps its complete original stdout/stderr in an evidence log.
# Failures still stop execution and print a bounded diagnostic excerpt.
import time
LOG_DIR = BASE / "evidence" / "runtime" / "notebook-logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
def run_logged(command, label, **kwargs):
    import subprocess as _subprocess
    start = time.perf_counter()
    path = LOG_DIR / (label + ".log")
    kwargs.pop("check", None)
    kwargs.pop("capture_output", None)
    kwargs.pop("stdout", None)
    kwargs.pop("stderr", None)
    kwargs.pop("text", None)
    # Performance target is advisory: capture timings, never abort a valid run.
    requested_timeout = kwargs.pop('timeout', None)
    with path.open("w", encoding="utf-8") as log:
        process = _subprocess.run(command, stdout=log, stderr=_subprocess.STDOUT,
                                  timeout=requested_timeout, **kwargs)
    duration = round(time.perf_counter() - start, 3)
    if process.returncode:
        tail = path.read_text(encoding="utf-8", errors="replace").splitlines()[-24:]
        print("COMMAND_FAIL", label, "exit", process.returncode, "seconds", duration, "raw_log", path)
        print("\n".join(tail))
        raise _subprocess.CalledProcessError(process.returncode, command)
    print("COMMAND_PASS", label, "seconds", duration, "raw_log", path)
    return process


## 2. Runtime configuration / Cấu hình runtime

**English**

Production Run All generates two independent 512×512 samples, four parallel technical 1024×1024 outputs, and eight creative images (14 images total). The optional 2048×1904 experiment is disabled by default; enabling it adds a fifteenth image and extra runtime. To enable it when you have more time or want to test high resolution, open the runtime configuration code cell and change `ZIMAGE_RUN_2000PX = False` to `ZIMAGE_RUN_2000PX = True` before Run All. No Kaggle Secret or external environment variable is needed. The high-resolution cell explicitly prints SKIPPED when disabled.

Other optional experiments (RUN_PAIR_THROUGHPUT, RUN_HIGH_RES, RUN_STRETCH_2560) stay disabled by default.

**Tiếng Việt**

Run All mặc định sinh hai ảnh 512×512 độc lập, bốn ảnh kỹ thuật 1024×1024 song song và tám ảnh creative (tổng 14 ảnh). Thử nghiệm 2048×1904 là tùy chọn, mặc định tắt; khi bật sẽ tạo ảnh thứ 15 và làm tăng thời gian chạy. Nếu có nhiều thời gian hoặc muốn kiểm thử độ phân giải cao, hãy mở cell cấu hình runtime và đổi `ZIMAGE_RUN_2000PX = False` thành `ZIMAGE_RUN_2000PX = True` trước khi Run All. Không cần Kaggle Secrets hoặc biến môi trường bên ngoài. Cell sẽ in trạng thái SKIPPED khi tắt.

Các thử nghiệm tùy chọn khác (RUN_PAIR_THROUGHPUT, RUN_HIGH_RES, RUN_STRETCH_2560) vẫn được tắt theo mặc định.


## Current-run evidence isolation / Cô lập bằng chứng của lần chạy hiện tại

**English**

Assign a unique run identifier and timestamp before inference. The gallery refuses images, manifests and benchmark files older than this run. Incomplete runs show an explicit error instead of historic results.

**Tiếng Việt**

Tạo mã phiên chạy và dấu thời gian duy nhất trước khi inference. Gallery từ chối ảnh, manifest và benchmark cũ hơn phiên này. Nếu chạy chưa hoàn tất, trang sẽ báo lỗi rõ ràng thay vì hiện kết quả cũ.

In [ ]:
# Create an isolated per-execution run context BEFORE generating images.
from pathlib import Path
import json, time, uuid
RUN_EVIDENCE_DIR = BASE / "evidence/showcase"
RUN_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
RUN_ID = "kaggle-" + uuid.uuid4().hex[:16]
RUN_STARTED_EPOCH = time.time()
RUN_CONTEXT = RUN_EVIDENCE_DIR / "run-context.json"
RUN_CONTEXT.write_text(json.dumps({"state":"STARTED","run_id":RUN_ID,"started_epoch":RUN_STARTED_EPOCH}, indent=2) + "\n")
print("CURRENT_RUN_ID", RUN_ID, "STARTED_EPOCH", RUN_STARTED_EPOCH)


In [ ]:
MODEL = Path("/kaggle/input/models/dangkhoa2016/tongyi-mai-z-image/pytorch/default/1")
assert MODEL.is_dir() and (MODEL / "model_index.json").is_file() and (MODEL / "transformer").is_dir(), ("Attach the Kaggle mirror via Add Input → Models (PyTorch/default/version 1) and verify: " + str(MODEL))
RUN_512 = True
RUN_1024 = True
RUN_PAIR_THROUGHPUT = False
RUN_HIGH_RES = False
RUN_STRETCH_2560 = False
# Change only this line to True to include the optional 2048x1904 image.
# Đổi thành True nếu muốn tạo thêm ảnh 2048x1904. No Secrets needed.
ZIMAGE_RUN_2000PX = False
assert isinstance(ZIMAGE_RUN_2000PX, bool), "Use True or False without quotation marks"
RUN_2000PX = ZIMAGE_RUN_2000PX
print("OPTIONAL_2048X1904", "ENABLED" if RUN_2000PX else "DISABLED")
# Lock the optional high-resolution choice to this notebook run for gallery verification.
current_context = json.loads(RUN_CONTEXT.read_text())
assert current_context["run_id"] == RUN_ID
current_context["run_highres"] = RUN_2000PX
RUN_CONTEXT.write_text(json.dumps(current_context, indent=2) + "\n")
STEPS = 28
CFG = 3.0
SEED = 42
PROMPT = "A cinematic photograph of a narrow old Japanese alley at blue hour after rain, warm lanterns glowing under wooden eaves, wet stone pavement reflecting amber light, small plants along the walls, subtle mist, deep perspective, realistic architecture, highly detailed, no people."
NEGATIVE_PROMPT = "blurry, distorted, low quality, malformed geometry, unreadable text, watermark, logo"
for path in [BASE / "cache", BASE / "evidence/runtime", BASE / "evidence/runs"]:
    path.mkdir(parents=True, exist_ok=True)
print("MODEL_OK", MODEL)
print("DEFAULTS", {"steps": STEPS, "cfg": CFG, "seed": SEED})


## 3. Runtime identity / Định danh runtime

**English**

Record the exact Python and library versions before inference. The runtime identity is shown before performance numbers because PyTorch/XLA and Diffusers behavior is version-sensitive.

**Tiếng Việt**

Ghi lại chính xác phiên bản Python và các thư viện trước khi inference. Runtime identity được hiển thị trước số hiệu năng vì hành vi PyTorch/XLA và Diffusers phụ thuộc phiên bản.


In [ ]:
import platform, importlib
mods = ["torch", "torch_xla", "diffusers", "transformers", "numpy", "PIL"]
print("Python", platform.python_version())
for name in mods:
    module = importlib.import_module(name)
    print(name, getattr(module, "__version__", "unknown"))
print("PJRT_DEVICE", os.environ.get("PJRT_DEVICE", "<kernel-env inherited by helper>"))


## 4. Model identity / Định danh model

**English**

Verify the local Kaggle mount contains the transformer, VAE, scheduler, tokenizer/text components, and configuration files. The notebook uses `local_files_only=True` through repository scripts and does not download or redistribute model weights.

**Tiếng Việt**

Xác minh Kaggle mount local có transformer, VAE, scheduler, tokenizer/text components và các file cấu hình. Notebook dùng `local_files_only=True` thông qua scripts của repository và không download hay redistribute model weights.


In [ ]:
required_components = ["transformer", "vae", "scheduler"]
missing = [name for name in required_components if not (MODEL / name).exists()]
assert not missing, f"Missing model components: {missing}"
print("MODEL_ROOT", MODEL)
for name in sorted(p.name for p in MODEL.iterdir()):
    print(" -", name)


## 5. Real TPU health check / Kiểm tra TPU thật

**English**

Run a real BF16 matrix operation through PyTorch/XLA and require exactly eight global TPU devices. This is **live execution evidence** from the current Kaggle session, not a cached repository result.

**Tiếng Việt**

Chạy phép toán ma trận BF16 thật qua PyTorch/XLA và yêu cầu đúng tám TPU devices toàn cục. Đây là **live execution evidence** của Kaggle session hiện tại, không phải kết quả cache trong repository.


In [ ]:
health = r"""import torch, torch_xla
import torch_xla.runtime as xr
xr.use_spmd(); d=torch_xla.device()
x=torch.arange(4096,device=d,dtype=torch.bfloat16).reshape(64,64)
y=(x@x.T).sum(); torch_xla.sync()
assert xr.global_runtime_device_count()==8
print('TPU_HEALTH_PASS',d,'global=',xr.global_runtime_device_count(),'addressable=',xr.addressable_runtime_device_count(),'value=',float(y.cpu()))
"""
r = subprocess.run(["python3", str(BASE/"run_with_kernel_env.py"), "python3", "-u", "-c", health], text=True, capture_output=True)
(LOG_DIR/"tpu-health.log").write_text((r.stdout or "") + (r.stderr or ""), encoding="utf-8")
print("TPU_HEALTH", "PASS" if r.returncode == 0 and "TPU_HEALTH_PASS" in r.stdout else "FAIL", "raw_log", LOG_DIR/"tpu-health.log")
if r.returncode: print((r.stderr or r.stdout)[-2500:])
assert r.returncode == 0 and "TPU_HEALTH_PASS" in (r.stdout or ""), "TPU health check failed"


## 6. Evidence policy and prompt suite / Chính sách evidence và bộ prompt

**English**

Three evidence classes are kept separate: **live execution evidence** is generated now; **accepted repository evidence** comes from frozen JSON/PNG artifacts already reviewed; **opt-in stretch evidence** documents expensive high-resolution paths that are reproducible but not rerun by default. The next cell loads the canonical accepted evidence through one tested adapter.

**Tiếng Việt**

Ba lớp evidence được tách rõ: **live execution evidence** được tạo trong session hiện tại; **accepted repository evidence** đến từ JSON/PNG đã freeze và review; **opt-in stretch evidence** mô tả các đường high-resolution tốn thời gian, có thể tái lập nhưng không rerun mặc định. Cell tiếp theo đọc canonical evidence qua một adapter đã có test.


In [ ]:
import sys
sys.path.insert(0, str(BASE))
from zimage_tpu.publication import (
    load_pair_throughput_rows, load_resolution_rows,
    load_showcase_rows, verify_showcase_assets,
)
showcase_rows = verify_showcase_assets(BASE)
throughput_rows = load_pair_throughput_rows(BASE)
resolution_rows = load_resolution_rows(BASE)
print("SHOWCASE_ACCEPTED", len(showcase_rows))
print("PAIR_BENCHMARK_ROWS", len(throughput_rows))
print("RESOLUTION_ROWS", len(resolution_rows))


## 7. Prepare deterministic prompt embeddings / Chuẩn bị prompt embeddings xác định

**English**

Prompt encoding is CPU-heavy, so a lightweight TPU heartbeat runs only during this host-side preparation. It is stopped before measured denoising and is never included in inference timing.

**Tiếng Việt**

Prompt encoding nặng CPU, vì vậy heartbeat TPU nhẹ chỉ chạy trong giai đoạn chuẩn bị phía host. Heartbeat được dừng trước denoising có đo thời gian và không bao giờ được tính vào inference timing.


In [ ]:
import json
EMB = BASE / "cache/prompt_embeds_publication.pt"
cmd = ["python3", str(BASE/"run_with_kernel_env.py"), "python3", "-u", str(BASE/"scripts/encode_prompt_cpu.py"),
       "--prompt", PROMPT, "--negative-prompt", NEGATIVE_PROMPT, "--output", str(EMB)]
run_logged(cmd, "prompt-encode")
print("EMBEDDINGS_READY", EMB, EMB.stat().st_size)


## 8. 512×512 live smoke generation / Sinh ảnh 512×512 trực tiếp

**English**

This is the fast end-to-end production smoke path: BF16 transformer inference on the 1×8 model mesh, CFG 3.0, negative prompt, CPU VAE decode, PNG output, and SHA256 evidence.

**Tiếng Việt**

Đây là đường smoke production end-to-end nhanh: transformer BF16 trên model mesh 1×8, CFG 3.0, negative prompt, CPU VAE decode, PNG output và SHA256 evidence.


## Lightweight notebook previews / Ảnh xem trước gọn nhẹ

**English**

Only the displayed image copies are resized to a maximum of 768 pixels. The full PNG files are preserved unmodified for the local gallery.

**Tiếng Việt**

Chỉ bản sao ảnh hiển thị được thu nhỏ với cạnh dài nhất 768 pixel. File PNG gốc giữ nguyên cho gallery trên máy TPU.

In [ ]:
from PIL import Image as _PreviewImage
from IPython.display import display as _native_display, Image as _InlineImage
from io import BytesIO as _BytesIO
MAX_INLINE_PX=768
def display(*items, **kwargs):
    compact=[]
    for item in items:
        if isinstance(item,_PreviewImage.Image):
            preview=item.convert("RGB")
            preview.thumbnail((MAX_INLINE_PX,MAX_INLINE_PX),_PreviewImage.Resampling.LANCZOS)
            buffer=_BytesIO()
            preview.save(buffer,format="JPEG",quality=85,optimize=True)
            compact.append(_InlineImage(data=buffer.getvalue(),format="jpeg"))
        else:
            compact.append(item)
    return _native_display(*compact,**kwargs)
print("INLINE_IMAGE_PREVIEW_MAX_EDGE",MAX_INLINE_PX)


In [ ]:
def run_single(width, height, slug, cfg_execution="batched", chunk=0, chunk_sync=False, seed_value=None):
    seed_value = SEED if seed_value is None else int(seed_value)
    denoise = ["python3", str(BASE/"run_with_kernel_env.py"), "python3", "-u", str(BASE/"run_zimage_spmd.py"),
        "--width", str(width), "--height", str(height), "--steps", str(STEPS), "--guidance", str(CFG),
        "--seed", str(seed_value), "--slug", slug, "--negative-mode", "explicit", "--cfg-execution", cfg_execution,
        "--embeddings", str(EMB)]
    if chunk: denoise += ["--attention-query-chunk-size", str(chunk)]
    if chunk_sync: denoise += ["--attention-chunk-sync"]
    run_logged(denoise, f"denoise-{slug}")
    run_dir = BASE / f"evidence/runs/run-{slug}-seed{seed_value}"
    decode = ["python3", str(BASE/"run_with_kernel_env.py"), "python3", "-u", str(BASE/"scripts/decode_vae.py"),
        "--latent", str(BASE/f"cache/latent_{slug}.pt"), "--run-dir", str(run_dir), "--slug", f"vae-{slug}",
        "--width", str(width), "--height", str(height)]
    run_logged(decode, f"vae-{slug}")
    return json.loads((BASE/f"evidence/runtime/{slug}.json").read_text()), json.loads((BASE/f"evidence/runtime/vae-{slug}.json").read_text())
if RUN_512:
    run512, vae512 = run_single(512, 512, "publication-512")
    print({"status": run512["status"], "denoise_s": run512["total_denoise_seconds"], "vae_s": vae512["vae_decode_seconds"], "sha256": vae512["sha256"]})
    run512_alt, vae512_alt = run_single(512, 512, "publication-512-alternate", seed_value=SEED + 1)
    assert run512_alt["status"] == "PASS" and vae512_alt["status"] == "PASS"
    assert vae512_alt["sha256"] != vae512["sha256"], "Technical verification images must be distinct"
    print("SECOND_INDEPENDENT_512_PASS", "seed", SEED + 1, "sha256", vae512_alt["sha256"])


## 9. TPU v5e-8 topology showcase / Showcase topology TPU v5e-8

**English**

This is the core TPU proof. One PJRT process controls all eight TPU v5e devices as a two-dimensional mesh: `replica=4`, `model=2`, with **global batch 4**. Each replica produces one sample while its Z-Image transformer is model-sharded across two TPU devices. The live probe below fails closed unless all eight devices are visible.

```text
Replica 0 → TPU 0 + TPU 1 → sample seed 42
Replica 1 → TPU 2 + TPU 3 → sample seed 43
Replica 2 → TPU 4 + TPU 5 → sample seed 44
Replica 3 → TPU 6 + TPU 7 → sample seed 45
```

**Tiếng Việt**

Đây là phần chứng minh TPU chính. Một PJRT process điều khiển toàn bộ tám TPU v5e theo mesh hai chiều: `replica=4`, `model=2`, với **global batch 4**. Mỗi replica tạo một sample trong khi transformer Z-Image của replica đó được model-shard trên hai TPU devices. Probe trực tiếp bên dưới sẽ fail-closed nếu không nhìn thấy đủ tám devices.


In [ ]:
import json as _json
TOPOLOGY_RUNNER = BASE / "scripts/run_zimage_tpu_batch4.py"
config_cmd = ["python3", str(TOPOLOGY_RUNNER), "--describe-config"]
topology_config = _json.loads(subprocess.check_output(config_cmd, text=True))
probe_path = BASE / "evidence/runtime/notebook-tpu-b4-mesh-probe.json"
probe_cmd = ["python3", str(BASE/"run_with_kernel_env.py"), "env", f"PYTHONPATH={BASE}", "python3", "-u", str(TOPOLOGY_RUNNER),
             "--mesh-probe-only", "--output-json", str(probe_path)]
run_logged(probe_cmd, "mesh-probe")
probe = _json.loads(probe_path.read_text())
assert probe["global_runtime_device_count"] == 8
assert probe["addressable_runtime_device_count"] == 8
assert probe["mesh_shape"] == [4, 2]
assert topology_config["batch_size"] == 4
print("TPU_TOPOLOGY_PASS", {"devices": 8, "mesh": probe["mesh_shape"], "axis_names": probe["axis_names"], "global_batch": 4})
for replica, seed in enumerate([42, 43, 44, 45]):
    print(f"Replica {replica} -> TPU {2*replica},{2*replica+1} -> seed {seed}")


## 10. Four concurrent 1024×1024 outputs / Bốn ảnh 1024×1024 đồng thời

**English**

The production path runs **four concurrent 1024** samples on the measured winning topology: one PJRT process, mesh `(replica=4, model=2)`, BF16 transformer, sequential CFG, 28 steps, guidance 3.0, seeds 42–45. The four latents are decoded on CPU and displayed as a 2×2 contact sheet so the notebook proves both TPU execution and visible output diversity.

**Tiếng Việt**

Đường production chạy **bốn ảnh 1024 đồng thời** trên topology thắng qua benchmark: một PJRT process, mesh `(replica=4, model=2)`, transformer BF16, CFG tuần tự, 28 steps, guidance 3.0, seeds 42–45. Bốn latent được decode trên CPU và hiển thị thành contact sheet 2×2 để notebook chứng minh cả thực thi TPU lẫn độ đa dạng đầu ra.


In [ ]:
from PIL import Image, ImageOps, ImageDraw
if RUN_1024:
    slug = "publication-tpu-b4-1024"
    cmd = ["python3", str(BASE/"run_with_kernel_env.py"), "env", f"PYTHONPATH={BASE}", "python3", "-u", str(TOPOLOGY_RUNNER),
           "--run", "--width", "1024", "--height", "1024", "--steps", str(STEPS), "--guidance", str(CFG),
           "--slug", slug, "--embeddings", str(EMB)]
    run_logged(cmd, "batch-1024")
    batch1024 = json.loads((BASE/f"evidence/dual/{slug}.json").read_text())
    assert batch1024["status"] == "PASS" and batch1024["mesh_shape"] == [4, 2]
    decoded = []
    for i, latent in enumerate(batch1024["latent_paths"]):
        run_dir = BASE / f"evidence/runs/run-{slug}-{i}"
        decode_slug = f"vae-{slug}-{i}"
        decode_cmd = ["python3", str(BASE/"run_with_kernel_env.py"), "python3", "-u", str(BASE/"scripts/decode_vae.py"),
                      "--latent", latent, "--run-dir", str(run_dir), "--slug", decode_slug, "--width", "1024", "--height", "1024"]
        run_logged(decode_cmd, f"vae-batch-1024-{i}")
        decoded.append(Image.open(run_dir/"output.png").convert("RGB"))
    thumbs = [ImageOps.fit(im, (512, 512)) for im in decoded]
    sheet = Image.new("RGB", (1024, 1024))
    for i, im in enumerate(thumbs): sheet.paste(im, ((i % 2) * 512, (i // 2) * 512))
    print("TPU_BATCH4_1024_PASS", {"images": 4, "denoise_s": batch1024["total_denoise_seconds"], "seeds": batch1024["seeds"], "mesh": batch1024["mesh_shape"]})
    display(sheet)


## Optional 2048×1904 inference / Inference 2048×1904 tùy chọn

**English**

This costly test uses one TPU replica with eight model shards and CPU VAE decoding. It is OFF by default (ZIMAGE_RUN_2000PX=False) to keep the production demo shorter. Enabling it adds a fifteenth image to the 14-image default gallery. For an additional real 2048×1904 render, change `ZIMAGE_RUN_2000PX = False` to `ZIMAGE_RUN_2000PX = True` in the runtime configuration code cell before Run All. If disabled, this cell prints SKIPPED without running inference.

**Tiếng Việt**

Thử nghiệm tốn thời gian này dùng một TPU replica với tám model shard và CPU VAE để giải mã. Mặc định TẮT (ZIMAGE_RUN_2000PX=False) nhằm rút ngắn thời gian demo production. Khi bật, gallery sẽ có 15 ảnh thay vì 14. Để sinh thêm ảnh 2048×1904 thực, đổi `ZIMAGE_RUN_2000PX = False` thành `ZIMAGE_RUN_2000PX = True` trong cell cấu hình runtime trước khi Run All. Khi tắt, cell in SKIPPED và không chạy inference.


In [ ]:
if RUN_2000PX:
    highrec,highvae=run_single(2048,1904,"publication-2048x1904",cfg_execution="sequential")
    assert highrec["status"]=="PASS" and highvae["status"]=="PASS"
    high_path=BASE/f"evidence/runs/run-publication-2048x1904-seed{SEED}/output.png"
    hi=Image.open(high_path).convert("RGB")
    assert hi.size==(2048,1904)
    print("SINGLE_REPLICA_2048x1904_PASS",{"model_shards":8,"resolution":hi.size,"denoise_s":highrec["total_denoise_seconds"],"vae_s":highvae["vae_decode_seconds"],"sha256":highvae["sha256"]})
    display(hi)
else:
    print("OPTIONAL_2048X1904_SKIPPED", "Set ZIMAGE_RUN_2000PX=True before Run All to enable.")


## 11. Eight-image creative showcase / Showcase tám ảnh

**English**

The accepted 1024×1024 showcase spans two stylized-3D scenes, two illustrations, and two photorealistic scenes. These are **accepted repository evidence**. Each image below is checked against its frozen SHA256 before display; no stored image is presented as a fresh render.

**Tiếng Việt**

Showcase 1024×1024 đã acceptance gồm hai cảnh stylized 3D, hai illustration và hai cảnh photoreal. Đây là **accepted repository evidence**. Mỗi ảnh bên dưới được kiểm SHA256 theo evidence đã freeze trước khi hiển thị; không ảnh lưu sẵn nào được mô tả như fresh render.


## Eight new Creative Showcase images / Tám ảnh Creative Showcase được sinh mới

**English**

Eight distinct 1024×1024 prompts are encoded after one text encoder load. Two real TPU inference batches generate four images each using a 4×2 mesh (four replicas, two model shards). A single CPU VAE load then decodes eight unique latent tensors. Every image is verified using its original PNG and SHA256. This reduces repeated initialization without claiming cached images as newly generated.

**Tiếng Việt**

Tám prompt riêng biệt ở độ phân giải 1024×1024 được mã hóa sau một lần tải text encoder. Hai batch inference TPU thật sinh bốn ảnh mỗi batch với mesh 4×2 (bốn replica, hai model shard). Sau đó CPU VAE được tải một lần để giải mã tám latent riêng biệt. Mỗi ảnh được xác minh bằng PNG gốc và SHA256. Cách này giảm khởi tạo lặp mà không nhận ảnh cache là ảnh mới sinh.


In [ ]:
import os,sys,subprocess,json
SHOWCASE_ROOT=BASE/'evidence/showcase'
assert (BASE/'scripts/showcase/run_creative.py').is_file()
run_logged([sys.executable,str(BASE/'scripts/showcase/prepare_prompts.py')], 'creative-prepare', cwd=BASE,env=os.environ|{'PYTHONPATH':str(BASE)})
run_logged([sys.executable,str(BASE/'scripts/showcase/encode_all_optimized.py')], 'creative-encode', cwd=BASE)
run_logged([sys.executable,str(BASE/'scripts/showcase/run_creative.py')], 'creative-generate', cwd=BASE)
fresh=json.loads((SHOWCASE_ROOT/'fresh-six-manifest.json').read_text())
extra=json.loads((SHOWCASE_ROOT/'creative-extra-acceptance.json').read_text())
assert fresh['status']=='PASS' and fresh['count']==6
assert extra['status']=='PASS' and extra['count']==2
creative_images=fresh['items']+extra['items']
assert len(creative_images)==8 and len({x['sha256'] for x in creative_images})==8
print('EIGHT_CREATIVE_IMAGES_VERIFIED',len(creative_images))


In [ ]:
from PIL import Image
import json
SHOWCASE_ROOT=(BASE/'evidence/showcase')
fresh=json.loads((SHOWCASE_ROOT/'fresh-six-manifest.json').read_text())
assert fresh['status']=='PASS' and fresh['count']==6
for row in creative_images:
    print('FRESH_CREATIVE',row['id'],'seed',row['seed'],'batch',row['batch'],'sha256',row['sha256'])
    print('PROMPT',row['prompt'])
    display(Image.open(row['png']))

## 12. Visual contact sheet / Contact sheet trực quan

**English**

A compact two-column contact sheet makes category consistency and visual diversity easy to review in one screen. It is derived from the six SHA-verified accepted PNGs and is not separate generation evidence.

**Tiếng Việt**

Contact sheet hai cột giúp người dùng đánh giá consistency theo category và độ đa dạng hình ảnh trong một màn hình. Nó được tạo từ sáu PNG accepted đã kiểm SHA và không phải generation evidence riêng biệt.


In [ ]:
from PIL import Image,ImageDraw
thumbs=[]
for row in creative_images:
    im=Image.open(row['png']).convert('RGB')
    im.thumbnail((360,360),Image.Resampling.LANCZOS)
    canvas=Image.new('RGB',(360,395),'white')
    canvas.paste(im,((360-im.width)//2,0))
    ImageDraw.Draw(canvas).text((10,370),row['id']+' seed '+str(row['seed']),fill='black')
    thumbs.append(canvas)
sheet=Image.new('RGB',(720,395*4),'white')
for i,im in enumerate(thumbs):
    sheet.paste(im,((i%2)*360,(i//2)*395))
CONTACT_SHEET=Path('/kaggle/working/zimage-tpu/evidence/showcase/creative-contact-sheet.jpg')
sheet.save(CONTACT_SHEET,quality=90)
print('FRESH_CREATIVE_CONTACT_SHEET',CONTACT_SHEET)
display(sheet)

## 13. Reference image integrity / Kiểm tra ảnh tham chiếu

**English**

Verify published reference image SHA256 checksums.

**Tiếng Việt**

Xác minh mã SHA256 công khai của ảnh tham chiếu.


In [ ]:
assert len(showcase_rows)==6
print("REFERENCE_HASHES_VERIFIED",len(verify_showcase_assets(BASE)))


## 14. Measured TPU scaling evidence / Evidence scaling TPU đã đo

**English**

The live 1024 showcase above is backed by accepted repository evidence, not a guessed topology. At 1024×1024, `4×2 / batch 4` completed four images in about 90.264 seconds (`0.04431 img/s`). Larger global batches 5–8 were tested and failed XLA compilation because of HBM pressure, making batch 4 the **maximum safe tested batch** for this topology at 1024. Resolution-aware measurements also show why one batch policy should not be hard-coded for every image size: 512 favors batch 8, while 768 and 1024 favor batch 4.

**Tiếng Việt**

Showcase 1024 trực tiếp phía trên được hậu thuẫn bởi accepted repository evidence, không phải topology chọn theo phỏng đoán. Ở 1024×1024, `4×2 / batch 4` hoàn thành bốn ảnh trong khoảng 90.264 giây (`0.04431 img/s`). Global batch 5–8 đã được thử và đều fail XLA compile do áp lực HBM, vì vậy batch 4 là **maximum safe tested batch** cho topology này ở 1024. Kết quả theo resolution cũng cho thấy không nên hard-code một batch policy cho mọi kích thước: 512 phù hợp batch 8, còn 768 và 1024 phù hợp batch 4.


In [ ]:
b4_1024=json.loads((BASE/"evidence/reference/production.json").read_text())
max_safe={"maximum_safe_batch":b4_1024["maximum_safe_tested_batch_1024"]}
assert b4_1024["mesh_shape"] == [4,2]
assert max_safe["maximum_safe_batch"] == 4
print("1024",{"topology":"4x2","batch":4,"seconds":round(b4_1024["total_denoise_seconds"],3)})
print("MAX_SAFE_1024",max_safe)
print("RESOLUTION_AWARE_POLICY",{"512":"batch 8","768":"batch 4","1024":"batch 4"})
print("768_B4_VS_B8",{"b4_img_s":b4_1024["batch4_768_images_per_second"],"b8_img_s":b4_1024["batch8_768_images_per_second"]})
print("512_B8",{"seconds":b4_1024["batch8_512_seconds"],"images":8})


## 15. High-resolution capability / Năng lực high-resolution

**English**

Accepted evidence covers standard sequential CFG through 2048×1904, query-chunked 2048×2048, and the 2560×2048 stretch path. These rows are **opt-in stretch evidence**: they are reproducible, but the public default avoids a rerun because 2560×2048 alone required roughly 41 minutes of denoising in the accepted run.

**Tiếng Việt**

Accepted evidence bao phủ sequential CFG chuẩn tới 2048×1904, query-chunked 2048×2048 và stretch path 2560×2048. Các dòng này là **opt-in stretch evidence**: có thể tái lập, nhưng public default không rerun vì riêng 2560×2048 đã cần khoảng 41 phút denoising trong accepted run.


In [ ]:
for row in resolution_rows:
    print(f"{row['resolution']:>10} | {row['mode']:<22} | denoise={row['denoise_s']:.3f}s | warm={row['warm_s']:.3f}s | {row['status']}")
if RUN_HIGH_RES:
    run_single(2048, 1904, "notebook-2048x1904", cfg_execution="sequential")
if RUN_STRETCH_2560:
    run_single(2560, 2048, "notebook-2560x2048", cfg_execution="sequential", chunk=4096, chunk_sync=True)


## 16. Reproducibility / Khả năng tái lập

**English**

The notebook ties visible outputs back to prompts, deterministic seeds, execution modes, frozen JSON, and SHA256 hashes. Shape-dependent XLA compilation means cold and warm timings must remain separate; latency is not expected to scale monotonically with pixel count.

**Tiếng Việt**

Notebook liên kết output nhìn thấy được với prompt, deterministic seed, execution mode, JSON đã freeze và SHA256. XLA compilation phụ thuộc tensor shape nên cold timing và warm timing phải tách riêng; latency không được kỳ vọng tăng đơn điệu theo số pixel.


In [ ]:
import hashlib
checks = {}
for rel in [
    "evidence/reference/showcase.json",
    "evidence/reference/production.json",
    "evidence/reference/throughput.json",
    "evidence/reference/resolution.json",
]:
    path = BASE / rel
    checks[rel] = hashlib.sha256(path.read_bytes()).hexdigest()
print(json.dumps(checks, indent=2))
print("SHOWCASE_HASHES_VERIFIED", len(verify_showcase_assets(BASE)))


## 17. Limitations / Giới hạn

**English**

Images require visual inspection. High-resolution generation can be expensive.

**Tiếng Việt**

Ảnh cần được kiểm tra trực quan. Sinh ảnh độ phân giải lớn có thể tốn thời gian.


In [ ]:
limits={"reference_images":len(showcase_rows),"maximum_safe_tested_batch_1024":max_safe["maximum_safe_batch"]}
print(json.dumps(limits,indent=2))


## Live full-resolution gallery / Gallery ảnh gốc trực tiếp

**English**

Start a static HTML image gallery from the current TPU machine, immediately before the final acceptance cell. All generated PNGs are preserved at full resolution. The gallery is accessible through a temporary Cloudflared Quick Tunnel, which stops when the session ends. The URL is public to anyone holding it; no credentials or private files are served.

**Tiếng Việt**

Khởi động gallery HTML tĩnh từ máy TPU hiện tại, ngay trước cell nghiệm thu cuối. Tất cả PNG sinh ra được giữ nguyên độ phân giải. Cloudflared Quick Tunnel tạo URL tạm thời, sẽ ngừng hoạt động khi phiên kết thúc. Bất kỳ ai có URL đều có thể xem ảnh, vì vậy không đưa token hoặc file riêng tư lên web.

In [ ]:
import sys
import json
import re
from IPython.display import display, Markdown

# Build HTML gallery from the current notebook run.
gallery_dir = BASE / "evidence/showcase/gallery-complete"
run_logged(
    [sys.executable, str(BASE / "scripts/showcase/gallery_runtime.py"),
     "--repo", str(BASE), "--output", str(gallery_dir)],
    "gallery-build", cwd=BASE,
)

complete_gallery = json.loads((gallery_dir / "manifest.json").read_text())
expected_count = 15 if RUN_2000PX else 14
assert complete_gallery["status"] == "PASS"
assert complete_gallery["count"] == expected_count
assert complete_gallery["run_id"] == RUN_ID
assert len({x["sha256"] for x in complete_gallery["items"]}) == expected_count
assert sum(x["group"] == "creative" for x in complete_gallery["items"]) == 8
assert all(x["provenance"] == "FRESH_INFERENCE" for x in complete_gallery["items"])
print("GALLERY_BUILD_PASS", expected_count, "images")

# Start HTTP server and Cloudflare Quick Tunnel.
run_logged(
    [sys.executable, str(BASE / "scripts/showcase/serve_gallery.py")],
    "gallery-serve", cwd=BASE, timeout=120,
)

# Accept only the fresh, verified URL for this run.
serve_log = (LOG_DIR / "gallery-serve.log").read_text(
    encoding="utf-8", errors="replace")
verified_urls = re.findall(
    r"^VERIFIED_GALLERY_URL\s+"
    r"(https://[a-z0-9-]+\.trycloudflare\.com)"
    r"\s+RUN_ID\s+(\S+)",
    serve_log, flags=re.MULTILINE,
)
assert len(verified_urls) == 1, "Expected exactly one verified Cloudflare URL"
gallery_url, gallery_run_id = verified_urls[0]
assert gallery_run_id == RUN_ID == complete_gallery["run_id"]

print()
print("=" * 60)
print("FULL_RES_GALLERY_READY")
print("GALLERY_URL:", gallery_url)
print("ORIGINAL_PNGS:", expected_count)
print("RUN_ID:", RUN_ID)
print("=" * 60)

display(Markdown(
    f"### 🖼️ Full-resolution Z-Image Gallery\n\n"
    f"**{expected_count} original PNG images**\n\n"
    f"**[Open HTML Gallery]({gallery_url}/#gallery)**\n\n"
    f"`{gallery_url}`"
))
print("CURRENT_RUN_GALLERY_COMPLETE", expected_count)


## 18. Final run summary / Tổng kết lượt chạy

**English**

Current-run output is distinct from reference benchmark data.

**Tiếng Việt**

Output lượt chạy hiện tại được phân biệt với dữ liệu benchmark tham chiếu.


In [ ]:
assert len(showcase_rows) == 6
assert max_safe["maximum_safe_batch"] == 4
assert b4_1024["global_runtime_device_count"] == 8
print("FINAL_VERDICT", {
    "tpu": "v5e-8 / all 8 devices in one PJRT process",
    "production_1024": "replica=4 model=2, global batch 4",
    "maximum_safe_tested_batch_1024": 4,
    "resolution_policy": "512=batch 8; 768=batch 4; 1024=batch 4",
    "high_resolution": "accepted through 2560×2048",
})

assert complete_gallery["run_id"] == RUN_ID and complete_gallery["status"] == "PASS"
assert complete_gallery["count"] == (15 if RUN_2000PX else 14)
assert len({v["sha256"] for v in complete_gallery["items"]}) == complete_gallery["count"]
assert all(v["provenance"] == "FRESH_INFERENCE" for v in complete_gallery["items"])
assert len(creative_images) == 8
print("FINAL_CURRENT_RUN_IMAGE_ACCEPTANCE_PASS", complete_gallery["count"])
